# MemTorch en Google Colab

Este notebook instala MemTorch 1.1.6 (CPU) y ejecuta dos ejemplos: la caracterización de un memristor VTEAM y una red memristiva pequeña.

**Cómo usarlo**

1. Ejecuta las celdas en orden, de arriba abajo (Shift + Enter).
2. No necesitas GPU: MemTorch se instala en su versión CPU.
3. Colab borra todo al desconectarse. En cada sesión nueva, vuelve a ejecutar la celda de instalación.

**Por qué se instala así.** Colab usa Python 3.12 o más nuevo, pero MemTorch solo existe para Python 3.7–3.9. Por eso creamos un entorno aparte con Python 3.9 en `/content/mt` y ejecutamos los scripts con ese Python, no con el del notebook.

## Paso 1: instalar (2–3 minutos)

In [ ]:
%%bash
set -e
MT=/content/mt
PY=$MT/bin/python

pip install -q uv
uv venv -q --python 3.9 $MT
uv pip install -q --python $PY torch==1.10.2+cpu torchvision==0.11.3+cpu -f https://download.pytorch.org/whl/torch_stable.html
# MemTorch declara el paquete obsoleto 'sklearn': instalamos scikit-learn y MemTorch con --no-deps
uv pip install -q --python $PY "numpy<1.24" pandas scipy scikit-learn matplotlib seaborn ipython lmfit "setuptools<70" patchelf
uv pip install -q --python $PY --no-deps memtorch-cpu==1.1.6

# Por si el sistema rechaza bibliotecas con "pila ejecutable" (glibc 2.41+)
SITE=$($PY -c 'import site; print(site.getsitepackages()[0])')
find "$SITE" -name '*.so*' -type f -exec $MT/bin/patchelf --clear-execstack {} \; 2>/dev/null || true

$PY -W ignore -c "import torch, memtorch; print('MemTorch listo | torch', torch.__version__)"

## Paso 2: preparar las gráficas

MemTorch intenta abrir ventanas para sus gráficas, y aquí no hay ventanas. Este pequeño módulo hace que cada gráfica se guarde como `grafica_1.png`, `grafica_2.png`, etc. Para usarlo, pon `import colab_plots` en la **primera línea** de tus scripts.

In [ ]:
%%writefile colab_plots.py
"""Guarda las gráficas de matplotlib como PNG en lugar de mostrarlas en una ventana."""
import glob
import os

import matplotlib
matplotlib.use("Agg")                  # dibujar sin ventana
import matplotlib.pyplot as plt

for viejo in glob.glob("grafica_*.png"):   # borrar las de ejecuciones anteriores
    os.remove(viejo)

_contador = [0]

def _guardar_en_vez_de_mostrar(*args, **kwargs):
    _contador[0] += 1
    nombre = f"grafica_{_contador[0]}.png"
    plt.savefig(nombre, dpi=120, bbox_inches="tight")
    plt.close("all")
    print("Gráfica guardada:", nombre)

plt.show = _guardar_en_vez_de_mostrar   # MemTorch llama a plt.show(); lo cambiamos por guardar

In [ ]:
# Celda auxiliar: muestra todas las gráficas guardadas. Ejecútala después de cada script.
import glob
from IPython.display import Image, display

def ver_graficas():
    archivos = sorted(glob.glob("grafica_*.png"), key=lambda n: int(n.split("_")[1].split(".")[0]))
    if not archivos:
        print("No hay gráficas todavía.")
    for archivo in archivos:
        print(archivo)
        display(Image(archivo))

## Paso 3: caracterizar un memristor

Un memristor VTEAM con variabilidad: sus resistencias se "sortean" con `StochasticParameter`, así que las gráficas cambian un poco en cada ejecución.

- **R_on** es la resistencia baja (encendido) y **R_off** la alta (apagado).
- `sigma` es cuánto varía R_on de un dispositivo a otro. Prueba 10 y 100.

In [ ]:
%%writefile caracterizacion.py
import colab_plots   # siempre primero, para guardar las gráficas
import memtorch

sigma = 50   # desviación estándar de R_on, en ohmios

reference_memristor = memtorch.bh.memristor.VTEAM
reference_memristor_params = {'time_series_resolution': 1e-10,
                              'r_off': memtorch.bh.StochasticParameter(loc=1000, scale=200, min=2),
                              'r_on': memtorch.bh.StochasticParameter(loc=200, scale=sigma, min=1)}

memristor = reference_memristor(**reference_memristor_params)
print("R_on:", float(memristor.r_on), "ohmios | R_off:", float(memristor.r_off), "ohmios")
memristor.plot_hysteresis_loop()
memristor.plot_bipolar_switching_behaviour()

In [ ]:
!/content/mt/bin/python -W ignore caracterizacion.py
ver_graficas()

## Paso 4: red ideal frente a red memristiva

El ejemplo de la guía: una capa `Linear(4, 3)` convertida a crossbars de memristores VTEAM. Con la semilla 0, el error medio debe salir alrededor de 0.0146.

In [ ]:
%%writefile ejemplo.py
import copy

import torch
import memtorch
from memtorch.mn.Module import patch_model
from memtorch.map.Parameter import naive_map
from memtorch.map.Input import naive_scale

# Paso 3: un memristor
memristor = memtorch.bh.memristor.VTEAM()
print("Resistencia baja  (R_on): ", memristor.r_on, "ohmios")
print("Resistencia alta (R_off):", memristor.r_off, "ohmios")

# Paso 4: la red ideal
torch.manual_seed(0)
red = torch.nn.Sequential(torch.nn.Linear(4, 3))
x = torch.rand(2, 4)
print("Entrada x:")
print(x)
print("Salida de la red ideal:")
print(red(x).detach())

# Paso 5: la red memristiva
red_memristiva = patch_model(
    copy.deepcopy(red),
    memristor_model=memtorch.bh.memristor.VTEAM,
    memristor_model_params={"time_series_resolution": 1e-10},
    module_parameters_to_patch=[torch.nn.Linear],
    mapping_routine=naive_map,
    transistor=True,
    programming_routine=None,
    tile_shape=(16, 16),
    max_input_voltage=0.3,
    scaling_routine=naive_scale,
    source_resistance=5,
    line_resistance=5,
    ADC_resolution=8,
)
red_memristiva.tune_()

# Paso 6: comparar
ideal = red(x).detach()
memristiva = red_memristiva(x).detach()
print("Salida de la red memristiva:")
print(memristiva)
error = (ideal - memristiva).abs()
print("Error absoluto:")
print(error)
print("Error medio:", round(error.mean().item(), 4))

In [ ]:
!/content/mt/bin/python -W ignore ejemplo.py

## Cómo hacer tus propios experimentos

1. Edita una celda que empieza con `%%writefile`, por ejemplo cambia `sigma = 50`.
2. Ejecuta esa celda (guarda el archivo) y luego la celda que lo ejecuta.

Para un script nuevo, crea una celda con `%%writefile mi_script.py` y ejecútalo con `!/content/mt/bin/python -W ignore mi_script.py`.

**Si algo falla:** `No such file or directory: /content/mt/bin/python` significa que la sesión se reinició y hay que repetir el Paso 1. Los errores de Python (`NameError`, `SyntaxError`) se explican en `GUIA.md`, en el repositorio.